In [9]:
!pip install googletrans==3.1.0a0


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [11]:
!pip install legacy-cgi


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [43]:
from googletrans import Translator

translator = Translator()

### Exercise 1: Machine Translation

In [44]:
# Original English text
text = "Hello, how are you?"

# Translate English to French
translation = translator.translate(text, src='en', dest='fr')

# Display the result
print(f"Original: {text}")
print(f"Translated: {translation.text}")

Original: Hello, how are you?
Translated: Hello, how are you?


In [31]:
def batch_translate(texts, source_lang='en', destination_lang='fr'):
    results = []

    # Validate the list and language codes
    if not isinstance(texts, list):
        raise TypeError("texts must be a list.")

    if source_lang not in LANGUAGES:
        raise ValueError(f"Invalid source language code: {source_lang}")

    if destination_lang not in LANGUAGES:
        raise ValueError(f"Invalid destination language code: {destination_lang}")

    for text in texts:
        result = {
            "original_text": text,
            "translated_text": None,
            "source_language": source_lang,
            "destination_language": destination_lang,
            "error": None
        }

        try:
            if not isinstance(text, str):
                raise TypeError("Each text must be a string.")

            if not text.strip():
                raise ValueError("Text cannot be empty.")

            translation = translator.translate(
                text,
                src=source_lang,
                dest=destination_lang
            )

            result["translated_text"] = translation.text

        except Exception as error:
            result["error"] = str(error)

        results.append(result)

    return results

In [32]:
sample_texts = [
    "Hello, how are you?",
    "I am learning NLP.",
    "Machine translation is fascinating."
]

batch_results = batch_translate(sample_texts)

for item in batch_results:
    print("Original:", item["original_text"])
    print("Translated:", item["translated_text"])
    print("Source language:", item["source_language"])
    print("Destination language:", item["destination_language"])
    print("Error:", item["error"])
    print("-" * 50)

Original: Hello, how are you?
Translated: Hello, how are you?
Source language: en
Destination language: fr
Error: None
--------------------------------------------------
Original: I am learning NLP.
Translated: I am learning NLP.
Source language: en
Destination language: fr
Error: None
--------------------------------------------------
Original: Machine translation is fascinating.
Translated: Machine translation is fascinating.
Source language: en
Destination language: fr
Error: None
--------------------------------------------------


### Exercise 2: Language Detection

In [33]:
# Text in German
text = "Guten Tag, wie geht es dir?"

# Detect the language
detected = translator.detect(text)

language_code = detected.lang
language_name = LANGUAGES.get(language_code, "unknown")

print(f"Detected language: {language_code} ({language_name})")
print(f"Confidence: {detected.confidence}")

Detected language: en (english)
Confidence: 1


In [ ]:
class IntegratedTranslator:
    """A simple system for language detection and machine translation."""

    def __init__(self):
        # Store the translator and available language names
        self.translator = Translator()
        self.languages = LANGUAGES

    def _validate_text(self, text):
        """Check that the input is a non-empty string."""
        if not isinstance(text, str):
            raise TypeError("Text must be a string.")

        if not text.strip():
            raise ValueError("Text cannot be empty.")

    def _validate_language(self, language_code):
        """Check that a language code exists in LANGUAGES."""
        if not isinstance(language_code, str):
            raise TypeError("Language code must be a string.")

        language_code = language_code.lower()

        if language_code not in self.languages:
            raise ValueError(f"Invalid language code: {language_code}")

        return language_code

    def detect_language(self, text):
        """Detect the language of a text.

        Parameters:
            text (str): Text whose language should be detected.

        Returns:
            dict: Original text, language code, language name, confidence,
                  and an error value if detection fails.
        """
        result = {
            "original_text": text,
            "language_code": None,
            "language_name": None,
            "confidence": None,
            "error": None
        }

        try:
            self._validate_text(text)
            detected = self.translator.detect(text)

            result["language_code"] = detected.lang
            result["language_name"] = self.languages.get(
                detected.lang, "unknown"
            )
            result["confidence"] = detected.confidence

        except Exception as error:
            result["error"] = str(error)

        return result

    def translate_text(self, text, destination_lang='fr', source_lang='auto'):
        result = {
            "original_text": text,
            "translated_text": None,
            "source_language": None,
            "source_language_name": None,
            "destination_language": destination_lang,
            "destination_language_name": None,
            "error": None
        }

        try:
            self._validate_text(text)

            destination_lang = self._validate_language(destination_lang)
            result["destination_language"] = destination_lang
            result["destination_language_name"] = self.languages[destination_lang]

            if source_lang == 'auto':
                detection = self.detect_language(text)

                if detection["error"] is not None:
                    raise RuntimeError(
                        "Language detection failed: " + detection["error"]
                    )

                source_lang = detection["language_code"]
            else:
                source_lang = self._validate_language(source_lang)

            result["source_language"] = source_lang
            result["source_language_name"] = self.languages.get(
                source_lang, "unknown"
            )

            translation = self.translator.translate(
                text,
                src=source_lang,
                dest=destination_lang
            )

            result["translated_text"] = translation.text

        except Exception as error:
            result["error"] = str(error)

        return result

    def process_multiple(self, texts, destination_lang='fr', source_lang='auto'):
        if not isinstance(texts, list):
            raise TypeError("texts must be a list.")

        results = []

        for text in texts:
            # Each text is processed separately so one error does not stop the batch
            result = self.translate_text(
                text,
                destination_lang=destination_lang,
                source_lang=source_lang
            )
            results.append(result)

        return results

In [35]:
integrated_translator = IntegratedTranslator()
print("Integrated translation system is ready.")

Integrated translation system is ready.


In [36]:
test_text = "Hello, how are you?"

# English to French
result_fr = integrated_translator.translate_text(
    test_text, destination_lang='fr', source_lang='en'
)

# English to Spanish
result_es = integrated_translator.translate_text(
    test_text, destination_lang='es', source_lang='en'
)

print("English to French:", result_fr)
print("English to Spanish:", result_es)

English to French: {'original_text': 'Hello, how are you?', 'translated_text': 'Hello, how are you?', 'source_language': 'en', 'source_language_name': 'english', 'destination_language': 'fr', 'destination_language_name': 'french', 'error': None}
English to Spanish: {'original_text': 'Hello, how are you?', 'translated_text': 'Hello, how are you?', 'source_language': 'en', 'source_language_name': 'english', 'destination_language': 'es', 'destination_language_name': 'spanish', 'error': None}


In [37]:
texts = [
    "Hello, how are you?",
    "Bonjour, comment allez-vous?",
    "Hola, ¿cómo estás?",
    "Guten Tag, wie geht es dir?"
]

multiple_results = integrated_translator.process_multiple(
    texts,
    destination_lang='fr',
    source_lang='auto'
)

for item in multiple_results:
    print("Original:", item["original_text"])
    print("Detected source:", item["source_language_name"],
          f"({item['source_language']})")
    print("Translated:", item["translated_text"])
    print("Error:", item["error"])
    print("-" * 60)

Original: Hello, how are you?
Detected source: english (en)
Translated: Hello, how are you?
Error: None
------------------------------------------------------------
Original: Bonjour, comment allez-vous?
Detected source: english (en)
Translated: Bonjour, comment allez-vous?
Error: None
------------------------------------------------------------
Original: Hola, ¿cómo estás?
Detected source: english (en)
Translated: Hola, ¿cómo estás?
Error: None
------------------------------------------------------------
Original: Guten Tag, wie geht es dir?
Detected source: english (en)
Translated: Guten Tag, wie geht es dir?
Error: None
------------------------------------------------------------


In [38]:
# Invalid language code
invalid_language_result = integrated_translator.translate_text(
    "Hello",
    destination_lang='invalid_code',
    source_lang='en'
)
print("Invalid language code:", invalid_language_result)

# Empty text
empty_text_result = integrated_translator.translate_text(
    "   ",
    destination_lang='fr',
    source_lang='en'
)
print("Empty text:", empty_text_result)

# Non-string input
non_string_result = integrated_translator.translate_text(
    12345,
    destination_lang='fr',
    source_lang='en'
)
print("Non-string input:", non_string_result)

# Batch with valid and invalid items
error_test_texts = [
    "Hello, how are you?",
    "",
    "Hola, ¿cómo estás?",
    123
]

error_batch_results = integrated_translator.process_multiple(
    error_test_texts,
    destination_lang='fr',
    source_lang='auto'
)

for item in error_batch_results:
    print(item)
    print("-" * 60)

Invalid language code: {'original_text': 'Hello', 'translated_text': None, 'source_language': None, 'source_language_name': None, 'destination_language': 'invalid_code', 'destination_language_name': None, 'error': 'Invalid language code: invalid_code'}
Empty text: {'original_text': '   ', 'translated_text': None, 'source_language': None, 'source_language_name': None, 'destination_language': 'fr', 'destination_language_name': None, 'error': 'Text cannot be empty.'}
Non-string input: {'original_text': 12345, 'translated_text': None, 'source_language': None, 'source_language_name': None, 'destination_language': 'fr', 'destination_language_name': None, 'error': 'Text must be a string.'}
{'original_text': 'Hello, how are you?', 'translated_text': 'Hello, how are you?', 'source_language': 'en', 'source_language_name': 'english', 'destination_language': 'fr', 'destination_language_name': 'french', 'error': None}
------------------------------------------------------------
{'original_text': ''

In [39]:
long_sentence = (
    "Natural language processing helps computers understand human language. "
    "Machine translation can convert text from one language into another. "
    "Language detection can identify the language of a sentence automatically. "
)

long_text = long_sentence * 12

print("Long text character count:", len(long_text))
print("Is the text longer than 1000 characters?", len(long_text) > 1000)

long_text_result = integrated_translator.translate_text(
    long_text,
    destination_lang='fr',
    source_lang='en'
)

print("Translation completed:", long_text_result["translated_text"] is not None)
print("Error:", long_text_result["error"])
if long_text_result["translated_text"] is not None:
    print("Translated text:", long_text_result["translated_text"])

Long text character count: 2568
Is the text longer than 1000 characters? True
Translation completed: True
Error: None
Translated text: Natural language processing helps computers understand human language. Machine translation can convert text from one language into another. Language detection can identify the language of a sentence automatically. Natural language processing helps computers understand human language. Machine translation can convert text from one language into another. Language detection can identify the language of a sentence automatically. Natural language processing helps computers understand human language. Machine translation can convert text from one language into another. Language detection can identify the language of a sentence automatically. Natural language processing helps computers understand human language. Machine translation can convert text from one language into another. Language detection can identify the language of a sentence automatically. Natural l

In [40]:
def main():
    # Initialize the integrated translator
    translator_system = IntegratedTranslator()

    # Single translation with automatic source detection
    text = "Hello, how are you?"
    result = translator_system.translate_text(
        text,
        destination_lang='fr',
        source_lang='auto'
    )

    print("STANDARD TRANSLATION")
    print("Original:", result["original_text"])
    print("Source language:", result["source_language_name"])
    print("Translated:", result["translated_text"])
    print("Error:", result["error"])

    # Batch translation with automatic source detection
    texts = [
        "Hello, how are you?",
        "Bonjour, comment allez-vous?",
        "Hola, ¿cómo estás?",
        "Guten Tag, wie geht es dir?"
    ]

    print("\nBATCH TRANSLATION")
    results = translator_system.process_multiple(
        texts,
        destination_lang='en',
        source_lang='auto'
    )

    for item in results:
        print("\nOriginal:", item["original_text"])
        print("Source language:", item["source_language_name"])
        print("Translated:", item["translated_text"])
        print("Error:", item["error"])


if __name__ == "__main__":
    main()

STANDARD TRANSLATION
Original: Hello, how are you?
Source language: english
Translated: Hello, how are you?
Error: None

BATCH TRANSLATION

Original: Hello, how are you?
Source language: english
Translated: Hello, how are you?
Error: None

Original: Bonjour, comment allez-vous?
Source language: english
Translated: Bonjour, comment allez-vous?
Error: None

Original: Hola, ¿cómo estás?
Source language: english
Translated: Hola, ¿cómo estás?
Error: None

Original: Guten Tag, wie geht es dir?
Source language: english
Translated: Guten Tag, wie geht es dir?
Error: None
